In [33]:
import os
import json 
import getpass
import logging
from pathlib import Path
from time import sleep

from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import BaseTool

from typing import TypedDict, Dict, List

from PyDI.io import load_xml, load_parquet, load_csv
from PyDI.profiling import DataProfiler

## Initialize

In [26]:
OUTPUT_DIR = "output/"
INPUT_DIR = "input/"

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [27]:
logging.basicConfig(filename= OUTPUT_DIR + 'agent.log',
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%H:%M:%S',
                    level=logging.DEBUG,
                    encoding='utf-8')

## Tools

In [98]:
class ProfileDatasetTool(BaseTool):
    name: str = "profile_dataset"
    description: str = """
        A tool that takes the path name of the dataset (csv, parquet, and xml) and performs data analysis. A JSON string is returned with the profile
        data.
    """

    def _run(self, *args, **kwargs) -> str:
        
        # Accept path either positionally or as a kwarg (e.g. "__arg1")
        path = None
        if args:
            # first positional arg
            path = args[0]
        elif kwargs:
            # sometimes the LLM -> function call maps the single arg to "__arg1"
            # or some other autogenerated key. If there's exactly one kwarg, take it.
            if len(kwargs) == 1:
                path = list(kwargs.values())[0]
            else:
                # if multiple kwargs provided, try to find a param named "path"
                path = kwargs.get("path") or kwargs.get("file_path") or None

        if not path or not isinstance(path, str):
            raise ValueError("ProfileDatasetTool requires a single path string argument.")

        # check file exists
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found at: {path}")

        ext = os.path.splitext(path)[1].lower()

        # load dataset according to extension
        if ext == ".parquet":
            df = load_parquet(path)
        elif ext == ".csv":
            df = load_csv(path)
        elif ext == ".xml":
            df = load_xml(path, nested_handling="aggregate")
        else:
            raise ValueError(f"Unsupported format: {ext}. Supported: .csv, .parquet, .xml")

        if df is None or getattr(df, "empty", False):
            # return a structured JSON error string (LLM will see this as content)
            return json.dumps({"error": f"Dataset at {path} loaded as empty or failed to load."})

        profiler = DataProfiler()
        profile = profiler.summary(df)

        # ensure to return a JSON string (your docstring promised JSON string)
        try:
            profile_json = json.dumps(profile, default=str)
        except Exception:
            # fallback: convert to str if not json-serializable
            profile_json = json.dumps({"profile_str": str(profile)})

        return profile_json


## Agents

In [140]:
class SimpleModelAgentState(TypedDict):
    datasets: list
    data_profiles: list
    target_shema: list

class SimpleModelAgent:
    
    def __init__(self, model, profileTool):
        # initialize logger
        self.logger = logging.getLogger()
        
        # prepare the StateGraph
        graph = StateGraph(SimpleModelAgentState)

        # create nodes
        graph.add_node("profile_data", self.profile_data)

        # create edges
        graph.add_edge("profile_data", END)

        graph.set_entry_point("profile_data")
        self.graph = graph.compile()

        # make tools available to the model
        self.tools = {profileTool.name: profileTool}
        self.model = model
        self.model = self.model.bind_tools(profileTool)
        
        # self.profileTool = profileTool

    def profile_data(self, state:SimpleModelAgentState):
        self.logger.info('----------------------- Entering profile_data -----------------------')

        system_prompt = """
            You are a data scientist tasked with the integration of several datasets.
            For each dataset path provided, call the tool `profile_dataset` with the path
            (one tool call per dataset). After receiving the profiling results, summarize
            the important findings for each dataset to the user.
        """
        
        datasets_list_str = "\n".join(state['datasets'])
        human_content = f"Please profile these datasets (one call per dataset):\n{datasets_list_str}"
        message = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]
        self.logger.info("Input Message:" + str(message))
        
        result = self.model.invoke(message)
        self.logger.info("RESULT:" + str(result))

        tool_calls = result.tool_calls

        results = []
        for t in tool_calls:
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                self.logger.info("adapt_pipeline: ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append((t['args']['__arg1'], str(result)))
          
        self.logger.info('Leaving profile_data')
        return {'data_profiles': results}

## Invoke Pipeline

In [141]:
# Initialize model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# define datasets
datasets = [
    INPUT_DIR + "datasets/kaggle_small.parquet",
    INPUT_DIR + "datasets/uber_eats_small.parquet",
    INPUT_DIR + "datasets/yelp_small.parquet"
]

# prepare agent
agent = SimpleModelAgent(llm, ProfileDatasetTool())

# invoke agent
result = agent.graph.invoke({"datasets": datasets})


kaggle_small:
  Rows: 10,000
  Columns: 20
  Total nulls: 17,719
  Null percentage: 8.9%
  Null counts per column:
    website: 1,975 (19.8%)
    phone_raw: 516 (5.2%)
    phone_e164: 1,075 (10.8%)
    address_line1: 510 (5.1%)
    address_line2: 8,620 (86.2%)
    street: 895 (8.9%)
    house_number: 950 (9.5%)
    city: 609 (6.1%)
    state: 1,088 (10.9%)
    postal_code: 1,205 (12.0%)
    rating: 276 (2.8%)

uber_eats_small:
  Rows: 10,000
  Columns: 17
  Total nulls: 21,395
  Null percentage: 12.6%
  Null counts per column:
    address_line1: 66 (0.7%)
    address_line2: 9,434 (94.3%)
    street: 329 (3.3%)
    house_number: 99 (1.0%)
    city: 113 (1.1%)
    state: 128 (1.3%)
    postal_code: 46 (0.5%)
    rating: 5,590 (55.9%)
    rating_count: 5,590 (55.9%)

yelp_small:
  Rows: 10,000
  Columns: 20
  Total nulls: 11,257
  Null percentage: 5.6%
  Null counts per column:
    phone_raw: 526 (5.3%)
    phone_e164: 526 (5.3%)
    address_line1: 84 (0.8%)
    address_line2: 9,938 (99.4

In [142]:
result['data_profiles']

[('input/datasets/kaggle_small.parquet',
  '{"rows": 10000, "columns": 20, "nulls_total": 17719, "nulls_per_column": {"kaggle380k_id": 0, "source": 0, "name": 0, "name_norm": 0, "website": 1975, "map_url": 0, "phone_raw": 516, "phone_e164": 1075, "address_line1": 510, "address_line2": 8620, "street": 895, "house_number": 950, "city": 609, "state": 1088, "postal_code": 1205, "country": 0, "latitude": 0, "longitude": 0, "categories": 0, "rating": 276}, "dtypes": {"kaggle380k_id": "string", "source": "object", "name": "object", "name_norm": "object", "website": "object", "map_url": "object", "phone_raw": "object", "phone_e164": "object", "address_line1": "object", "address_line2": "object", "street": "object", "house_number": "object", "city": "object", "state": "object", "postal_code": "object", "country": "object", "latitude": "float64", "longitude": "float64", "categories": "object", "rating": "float64"}}'),
 ('input/datasets/uber_eats_small.parquet',
  '{"rows": 10000, "columns": 17, 